# 第 3 章 · Agent 详解：指令、参数与结构化输出

> **你已会**：第 1 章的 Agent / Runner / Session / Event。
>
> **读完后你能**：① 用参数控制 Agent 的行为边界；② 用状态插值写出动态 `instruction`；③ 用 `output_key` 在步骤之间传值；④ 在 DeepSeek 上稳定拿到可校验的 JSON。
>
> **本章不碰**：工具调用、多智能体、回调。分别在第 3、4、6 章。

**怎么读**

- **主线（约 40 分钟）**：§1 参数表 → §3.1 `instruction` → §4 `output_key` → §6 结构化输出。
- **深读**：§1 的 `mode` 说明、文末的 LangChain 对照（折叠）。
- 参数表不用背。先知道"有这些旋钮"，用的时候回来查。

---

## 1. Agent 的参数全景

`Agent` 是 `LlmAgent` 的别名。参数很多，但常用的就那几个。先看全貌：

| 参数 | 作用 | 什么时候会用到 |
|---|---|---|
| `name` | 智能体唯一标识 | 永远 |
| `model` | 底层模型 | 永远 |
| `instruction` | 人设、规则、输出格式 | 永远 |
| `description` | 一句话能力描述，供上级 Agent 路由 | 第 5 章 |
| `tools` | 可调用的工具 | 第 4 章 |
| `sub_agents` | 下级 Agent | 第 5 章 |
| `output_key` | 把最终回复写入 Session 状态 | 步骤间传递结果 |
| `output_schema` | 要求结构化输出 | 需要 JSON 时（见 §6 的 DeepSeek 限制） |
| `generate_content_config` | 温度、最大输出长度等 | 需要控制风格或成本时 |
| `before/after_*_callback` | 生命周期钩子 | 第 7 章 |
| `mode` | 对话 / 任务 / 单轮 | 第 8 章做工作流节点时 |
| `include_contents` | 是否把历史带进请求 | 无状态单轮任务可设 `none` 省 token |
| `planner` / `code_executor` | 规划器 / 代码执行器 | 进阶，本教程不展开 |

> 📌 **架构上的一个事实**：`Agent.model` 接受的是 `BaseLlm` 接口的实例（例如 `LiteLlm`），而不是某个具体厂商的客户端。ADK 因此不把运行时绑死在单一模型上。本教程只使用 `LiteLlm` + DeepSeek 这一条路径。

<details>
<summary><b>深入：<code>mode</code> 的默认值为什么"看场合"</b></summary>

`mode` 默认为 `None`，实际行为随使用方式而定：作为子 Agent 时默认 `chat`，作为工作流节点时默认 `single_turn`。

两者的真正区别不是"参不参与对话"，而是**输入从哪来**：

| 模式 | 输入来源 |
|---|---|
| `chat` | 会话历史（session events） |
| `single_turn` | 上游节点直接传入的 `node_input` |

所以把 `chat` 模式的 Agent 接在别的节点后面时，ADK 在构造工作流图阶段就会拒绝——它拿不到上游数据。第 8 章会真正用上这个区别。

</details>

---

## 2. DeepSeek 是怎么接进来的

ADK 原生字符串模型名走 Google 的模型路径。本教程统一传入一个 `LiteLlm` 对象：

```python
from google.adk.models.lite_llm import LiteLlm

model = LiteLlm(model="deepseek/deepseek-chat")
```

这里只有一条规则需要记：**`provider/model`**。

- `provider` 是 `deepseek`，LiteLLM 据此把请求路由到 DeepSeek；
- 同时它会读取环境变量 `DEEPSEEK_API_KEY`；
- `model` 部分是具体模型名 `deepseek-chat`。

这条路径不需要 `GOOGLE_API_KEY`，也不需要任何 Google Cloud 凭证。

> ⚠️ **唯一需要额外注意的地方**：DeepSeek 的 API 与 Gemini 并不完全等价。最典型的是结构化输出——DeepSeek 目前只支持 `response_format={"type": "json_object"}`，不支持 `json_schema`。这一点会直接影响 §6 的写法。

先用第 1 章的方式重建运行环境：

In [1]:
import os
assert os.environ.get("DEEPSEEK_API_KEY"), "请先设置 DEEPSEEK_API_KEY"

from google.adk.agents import Agent
from google.adk.models.lite_llm import LiteLlm
from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService
from google.genai import types

APP, USER = "adk_ch02", "student"

async def run_once(agent, query, session_id="demo", verbose=False):
    """通用一次性问答：新建 Runner+Session，返回最终文本（可选打印事件流）。"""
    ss = InMemorySessionService()
    await ss.create_session(app_name=APP, user_id=USER, session_id=session_id)
    runner = Runner(agent=agent, app_name=APP, session_service=ss)
    # 输入 msg
    msg = types.Content(role="user", parts=[types.Part(text=query)])
    final = ""
    async for ev in runner.run_async(user_id=USER, session_id=session_id, new_message=msg):
        # verbose(啰嗦) 将打印所有event
        if verbose and ev.content and ev.content.parts:
            for p in ev.content.parts:
                if p.text: print(f"  [event:{ev.author}] {p.text[:80]}")
        if ev.is_final_response() and ev.content and ev.content.parts:
            final = ev.content.parts[0].text
    # 顺便返回 session 便于查看状态
    session = await ss.get_session(app_name=APP, user_id=USER, session_id=session_id)
    return final, session

print("✅ 运行环境就绪")


✅ 运行环境就绪


---
## 3. `instruction`：不止是一段静态文字

### 3.1 基础用法：角色 + 规则 + 输出格式

一段能用的 `instruction` 通常有三块：

1. **角色**：你是谁；
2. **行为准则**：遇到什么情况怎么做；
3. **输出要求**：格式、长度、语气。

先猜：**如果只写"你是一位文学老师"，不写任何格式要求，回答会长什么样？** 运行后对照你的猜测。


In [2]:
poet = Agent(
    name="poet",
    model=LiteLlm(model="deepseek/deepseek-chat"),
    instruction="""你是一位精通中国古典诗词的文学老师。
行为准则：
1. 赏析诗词时先给白话翻译，再给艺术点评；
2. 点评不超过 100 字；
3. 结尾推荐一首风格相近的诗。
""",
    description="中国古典诗词赏析老师",
)

text, _ = await run_once(poet, "赏析一下「大漠孤烟直，长河落日圆」。")
print(text)


**白话翻译**：浩瀚沙漠中一缕孤烟笔直升起，黄河尽头一轮落日浑圆低垂。

**艺术点评**：一“直”一“圆”，以极简线条勾勒塞外苍茫，构图如画，动静相衬，尽显雄浑壮阔之美。

**推荐**：王维《使至塞上》全诗可参读；风格相近者荐王之涣《凉州词》“黄河远上白云间，一片孤城万仞山”。


### 3.2 状态插值：让 instruction "活"起来

`instruction` 里的 `{key}` 会在**发送给模型之前**，从 Session 状态里取值并替换。

```python
instruction="你正在和 {username} 对话，TA 的母语是 {native_lang}。"
```

状态值在创建会话时通过 `initial_state` 写入。于是同一个 Agent 定义，可以对不同用户呈现不同人设。

**先猜：如果状态里没有 `username`，这段 instruction 会原样发给模型，还是报错？** 运行后检查最终回复。

In [3]:
greeter = Agent(
    name="greeter",
    model=LiteLlm(model="deepseek/deepseek-chat"),
    instruction="你正在和 {username} 对话，TA 的母语是 {native_lang}。请用对方的母语打招呼，并自我介绍。",
    description="多语言问候助手",
)

# 预置 session 状态：initial_state 会在创建会话时写入
ss = InMemorySessionService()

await ss.create_session(app_name=APP, user_id=USER, session_id="i18n",
                        state={"username": "小明", "native_lang": "中文"}) # ← 关键: 这里使用 state:dict的方式传入参数
runner = Runner(agent=greeter, app_name=APP, session_service=ss)
msg = types.Content(role="user", parts=[types.Part(text="开始吧")])
async for ev in runner.run_async(user_id=USER, session_id="i18n", new_message=msg):
    if ev.is_final_response():
        print(ev.content.parts[0].text)


你好，小明！我是多语言问候助手，很高兴认识你。


> ⚠️ **三个容易踩的点**：
>
> 1. `{key}` 是**指令注入**，替换发生在调用模型之前。改了状态之后，下一轮才会生效；
> 2. **键不存在时会直接抛 `KeyError: Context variable not found`**，不是原样保留，也不会自动补默认值；
> 3. 确实可能缺失的字段，写成 **`{key?}`**（问号表示可选）：缺失或值为 `None` 时替换成空字符串，不会中断运行。
>
> 所以"可选信息"要在 instruction 里显式用 `?` 标出来，并约定空值时的行为。

---

## 4. `output_key`：把回复存进 Session 状态

默认情况下，Agent 的回复只出现在事件流里。

加上 `output_key` 后，最终文本会**自动写入 `session.state[output_key]`**。后续的 Agent、工具或你自己的代码都能直接读取。

这是多智能体"接力赛"里传递接力棒的标准动作（第 5 章会大量使用）。

**先猜：不加 `output_key`，上一步的回复还能从 Session 状态里直接取到吗？**

In [4]:
summarizer = Agent(
    name="summarizer",
    model=LiteLlm(model="deepseek/deepseek-chat"),
    instruction="把用户给的文本总结成一句话。",
    description="一句话总结器",
    output_key="summary",   # ← 关键：最终回复存入 state["summary"]
)

text, session = await run_once(summarizer, "人工智能正在深刻改变软件开发的方式，从代码补全到自动测试，再到今天的智能体协同编程，开发者的角色正从『写代码的人』转变为『指挥智能体的人』。")
print("Agent 回复：", text)
print("─" * 50)
print("Session 状态中已归档：", session.state.get("summary"))


Agent 回复： 人工智能正推动开发者从写代码转向指挥智能体协同编程。
──────────────────────────────────────────────────
Session 状态中已归档： 人工智能正推动开发者从写代码转向指挥智能体协同编程。


**如果不加 `output_key`, 回复的内容也可以直接从 `final_response` 中获取, 但无法从Session的状态中获取到**

---

## 5. `generate_content_config`：控制随机性与长度

温度（`temperature`）控制随机性：

- `0` 附近：更稳定、更确定，适合事实型任务；
- `1` 以上：更发散，适合创意型任务。

ADK 用 `GenerateContentConfig` 表达这些生成参数：

```python
generate_content_config=GenerateContentConfig(
    temperature=1.2,
    max_output_tokens=500,
)
```

> 📌 **不要只看单次输出下结论。** 同一个提示词至少跑三次，才能看出"稳定"还是"碰巧"。第 2 题练习就是让你用数据回答这个问题。

In [5]:
from google.genai.types import GenerateContentConfig

creative = Agent(
    name="creative",
    model=LiteLlm(model="deepseek/deepseek-chat"),
    instruction="给产品起 3 个中文名，每个附一句 slogan。",
    description="品牌创意命名师",
    generate_content_config=GenerateContentConfig(temperature=1.2, max_output_tokens=500),
)

text, _ = await run_once(creative, "产品：一款帮助程序员管理久坐健康的 App。")
print(text)


以下是为这款产品构思的 3 个中文名及对应 slogan：

1. **坐忘** — 把久坐抛到脑后，把健康留给身体。
2. **码上动** — 敲完这段代码，就起身动一动。
3. **腰码** — 程序员的腰，也是需要 commit 的资产。

命名思路简述：「坐忘」取庄子“坐忘”之意，反用为“忘掉久坐”；「码上动」谐音“马上动”，兼扣“代码”意象；「腰码」谐音“要嘛”，把腰与代码并置，制造记忆点。


---

## 6. 结构化输出：在 DeepSeek 上怎么做

业务系统要的往往不是"一段话"，而是**一个能入库的 JSON**。

ADK 提供了 `output_schema` 参数，但它依赖底层 API 的 `json_schema` 模式。

**DeepSeek 目前不支持这个模式**，直接传 `output_schema` 会报 `This response_format type is unavailable now`。这是模型方的限制，不是 ADK 的 bug。

使用 `output_schema` 的一个标准示例为:

```python
from pydantic import BaseModel, Field
from google.adk.agents import Agent
from google.adk.models.lite_llm import LiteLlm

class BookReview(BaseModel):
    title: str = Field(description="书名")
    rating: int = Field(description="评分，1-5 的整数")
    keywords: list[str] = Field(description="3 个关键词")
    one_line: str = Field(description="一句话点评")

reviewer = Agent(
    name="reviewer",
    model="gemini-3.5-flash",                      # Gemini 原生
    # model=LiteLlm(model="openai/gpt-4.1-mini"),  # 任何支持 json_schema 的 provider
    instruction="你是书评编辑，按 schema 输出书评。",
    output_schema=BookReview,
    output_key="review",                           # 可选：不写就只从事件里拿 JSON 字符串
)
```

以上面的代码为例子, ADK在发送给模型的请求中会多出一个内容:

```json
{
    "response_format": {
        "type": "json_schema",
        "json_schema": 
        {
            "name": "BookReview", 
            "strict": true,
            "schema": {
                "properties": {...}, 
                "required": ["title","rating","keywords","one_line"],
                "type": "object", 
                "additionalProperties": false
            }
        }
    }
}
```

模型返回只有一段 JSON 文本, 此后有ADK使用 `pydantic` 进行校验, 若模型违约则抛出 `pydantic.ValidationError`，整轮运行直接失败.

因此，`output_schema` 只在请求里增加一个约束开关，并非框架替你校验；违反该约束时会抛异常，最终校验仍由 ADK 在收到回复后完成.

----
所以在 DeepSeek 上，我们采用一套等价且更通用的工程做法：

| 环节 | 用什么保证 |
|---|---|
| 输出是 JSON | `response_format={"type": "json_object"}` |
| 字段齐全、类型正确 | Pydantic 模型校验 |
| 业务规则（长度、取值范围、关键词） | 自己写检查函数 |

> 📌 **结构化输出 = 格式约束 + 字段校验 + 业务检查。** 三段缺一不可。模型原生支持只是把第一段做到了接近 100%，并不替你做后两段。

下面用 Pydantic 定义一个书评结构，让模型按契约输出，再在本地校验：


In [8]:
from pydantic import BaseModel, Field
import json, re

class BookReview(BaseModel):
    title: str = Field(description="书名")
    rating: int = Field(description="评分，1-5 的整数")
    keywords: list[str] = Field(description="3 个关键词")
    one_line: str = Field(description="一句话点评")

reviewer_ds = Agent(
    name="reviewer_ds",
    model=LiteLlm(model="deepseek/deepseek-chat"),
    instruction="""你是书评编辑。严格输出如下 JSON（不要输出任何其他文字、不要用 markdown 代码块）：
{"title": "书名", "rating": 1到5的整数, "keywords": ["关键词1", "关键词2", "关键词3"], "one_line": "一句话点评"}""",
    description="书评结构化生成器（DeepSeek 兼容版）",
    output_key="review_json",
)

text, session = await run_once(reviewer_ds, "帮我分析一下《三体》这本书。")
print("原始输出：", text)

# 防御性解析：剥离可能的 markdown 围栏后用 Pydantic 校验
cleaned = re.sub(r"^```(json)?|```$", "", text.strip(), flags=re.M).strip()

review = BookReview(**json.loads(cleaned))
"""
为什么这里要使用**
1. json.loads(cleaned) -> 把字符串变成字典 
{"title": "三体", "rating": 5, "keywords": ["科幻", "宇宙社会学", "硬核物理"], "one_line": "宏大冷酷的宇宙史诗…"}

2. ** 把 dict 的每个键值对摊开成一串关键字参数：
BookReview(**{"title": "三体", "rating": 5}) -> BookReview(title="三体", rating=5)

这里整体等价于:
data = json.loads(cleaned)
review = BookReview(
    title=data["title"],
    rating=data["rating"],
    keywords=data["keywords"],
    one_line=data["one_line"],
)
"""
print("─" * 50)
print(f"✅ 解析成功：{review.title} | 评分 {review.rating}/5 | 关键词 {review.keywords}")


原始输出： {"title": "三体", "rating": 5, "keywords": ["硬科幻", "黑暗森林", "文明博弈"], "one_line": "以扎实的科学想象和冷酷的宇宙社会学，重新定义中文科幻高度的史诗开篇。"}
──────────────────────────────────────────────────
✅ 解析成功：三体 | 评分 5/5 | 关键词 ['硬科幻', '黑暗森林', '文明博弈']


> ⚠️ **`output_schema` 的一个框架级限制**（即使将来模型支持了也仍然存在）：启用它时，该 Agent 不能同时使用工具，也不能向其他 Agent 转移——框架要把输出通道完全锁定为结构化结果。遇到这种需求，标准解法是**分两个阶段**：第一阶段查信息，第二阶段只做结构化输出。练习 3 会让你设计这个边界。

<details>
<summary><b>深入：与 LangChain 的结构化输出对照</b></summary>

LangChain 的 `llm.with_structured_output(Model)` 可以改走**工具调用通道**，因此它在 DeepSeek 上可用。同样是结构化输出，两个框架的底层通道不同，兼容性和失败模式也不同。

值得记住的不是"谁更好"，而是：**任何框架的结构化输出，最终都要落到"模型支持哪种通道 + 你如何校验"这两个问题上。**

</details>

---

## 7. `description`：为多智能体埋下伏笔

`description` 在 Agent 自己运行时几乎不起作用。它的舞台在第 5 章：父 Agent 决定"把任务转交给哪个子 Agent"时，读的就是每个子 Agent 的 `description`。

因此它应该像一份**给调度系统看的岗位简介**：写清擅长什么、什么情况下该被选中。

```python
# ❌ 说了等于没说
description="一个很有用的助手"

# ✅ 说清了边界和触发条件
description="处理订单退款相关的请求，包括查询退款进度和发起退款申请"
```

> ✍️ 判断标准：把描述给另一个人看，他能否判断"这个问题该不该找它"。

---

<details>
<summary><b>深入：与 LangChain / LangGraph 对照</b></summary>

| ADK 概念 | LangChain / LangGraph 对应 | 差异 |
|---|---|---|
| `Agent(instruction=...)` | `create_agent(model, system_prompt=...)` | 几乎一一对应 |
| `LiteLlm(model="deepseek/...")` | `ChatOpenAI(base_url=...)` 等 | 接法不同，目标一致 |
| `output_key` 写入 state | 节点函数返回 `{"key": value}` | ADK 声明式一行；LangGraph 显式但灵活 |
| `generate_content_config` | 模型构造参数 | ADK 与 Agent 绑定，LangChain 与模型绑定 |
| instruction `{state}` 插值 | `ChatPromptTemplate` 变量 | ADK 直接读 Session state，少一层模板 |

</details>

---

## 📌 用五句话复述本章

1. **参数不用背**，先知道 `instruction` / `output_key` / `description` 各自负责什么。
2. **`instruction` 的 `{key}` 是调用模型前的状态替换**，缺值时不会自动补默认值。
3. **`output_key` 把最终回复写进 Session 状态**，是多 Agent 接力的基础。
4. **DeepSeek 不支持 `json_schema`**，所以结构化输出用 `json_object` + Pydantic 校验 + 业务检查三段式。
5. **`description` 是写给调度者看的**，要写清边界和触发条件。

> ➡️ 下一章：[04-工具系统](04-工具系统.ipynb) —— 让 Agent 从"会说"进化到"会做"。


---

## 🧪 本章练习

### 1. 参数实验：让差异可观察（基础）

固定同一提示词，只改 `generate_content_config`。

- [ ] 至少设置两组不同参数（例如低温 + 短输出、高温 + 长输出）
- [ ] 每组运行三次
- [ ] 用表格记录输出稳定性、格式遵循度、长度和主观质量
- [ ] 说明哪组适合事实型助手、哪组适合创意型助手

### 2. 动态 instruction 与状态接力（进阶）

做一个客服分诊 Agent。

- [ ] 从 Session State 读取用户等级、语言、产品类型
- [ ] 动态改变语气与处理规则
- [ ] 用 `output_key` 保存标准化分诊结论
- [ ] 覆盖"缺少状态值 / 普通用户 / VIP 用户"三种情况
- [ ] 确认状态中保留了下一 Agent 可用的结果

### 3. 结构化输出的工程边界（进阶）

为"会议行动项"设计输出契约。

- [ ] Pydantic Schema 至少包含负责人、任务、截止日期、优先级、风险提示
- [ ] 用 `json_object` + instruction 契约生成结果，并在本地校验
- [ ] 设计两阶段方案，解决"结构化输出不能同时用工具/转移"的限制
- [ ] 写清两个阶段之间的数据契约

### 4. 编写可路由的岗位说明（工程）

为退款、物流、技术支持、兜底四个子 Agent 写 `name`、`description` 和关键 `instruction`。

- [ ] 至少构造八条容易混淆的用户请求
- [ ] 人工评审 description 是否足以支持正确路由
- [ ] 把失败样例整理成回归集，第 5 章直接复用

### 5. 把 instruction 变成可测试的契约（工程）

选一个本章的 Agent，为它写一组离线检查。

- [ ] 至少覆盖三种输入：正常、边界、应该拒绝的请求
- [ ] 用代码断言回复的结构和关键字段，而不是"看起来不错"
- [ ] 说明哪些检查是确定性的、哪些依赖模型输出，需要多跑几次
- [ ] 记录一次失败样例，并写出你会怎么改 instruction